# Task 2: Quantitative Analysis with TA-Lib and PyNance

This notebook loads historical OHLCV stock-price data, cleans and validates numeric fields, computes technical indicators, applies PyNance-style financial metrics, and visualizes how indicators relate to price action.

## Data Requirements

Place a historical stock-price CSV in `../data/raw/`. The notebook accepts files such as `stock_prices.csv`, `historical_stock_prices.csv`, or `prices.csv`. Required columns are `Date`, `Open`, `High`, `Low`, `Close`, and `Volume`; `Adj Close` is recommended for return calculations. A `Stock` column is optional but recommended when multiple tickers are stored in one file.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.technical_indicators import (
    add_indicators_by_stock,
    clean_price_data,
    compute_pynance_metrics,
    load_price_data,
    missing_value_report,
)

sns.set_theme(style="whitegrid", palette="Set2")
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Prepare and Validate Price Data

The loader standardizes column names, parses `date`, converts OHLCV fields to numeric values, and uses `adj_close` for return calculations when available.

In [ ]:
raw_prices = load_price_data(PROJECT_ROOT / "data" / "raw")
raw_prices.head()

In [ ]:
print(f"Rows loaded: {len(raw_prices):,}")
print(f"Date range: {raw_prices['date'].min()} to {raw_prices['date'].max()}")
if 'stock' in raw_prices.columns:
    print(f"Tickers: {raw_prices['stock'].nunique():,}")

missing_before = missing_value_report(raw_prices)
missing_before.to_frame("missing_values")

In [ ]:
prices = clean_price_data(raw_prices)
missing_after = missing_value_report(prices)
missing_after.to_frame("missing_values_after_cleaning")

## 2. Compute Technical Indicators with TA-Lib

The helper functions call TA-Lib for SMA, EMA, RSI, and MACD when TA-Lib is installed. If TA-Lib is unavailable in a local environment, equivalent pandas calculations are used as a reproducible fallback so the workflow remains executable.

In [ ]:
indicators = add_indicators_by_stock(prices)
indicator_columns = [column for column in indicators.columns if column.startswith(('sma_', 'ema_', 'rsi_')) or column.startswith('macd')]
indicators[['date', 'adj_close', 'daily_return', *indicator_columns]].tail()

## 3. Apply PyNance for Financial Metrics

The metrics table summarizes cumulative return, annualized volatility, max drawdown, and a zero-risk-free-rate Sharpe ratio. The `pynance_available` flag records whether PyNance was importable in the active kernel.

In [ ]:
if 'stock' in indicators.columns:
    metrics = (
        indicators.groupby('stock', group_keys=False)
        .apply(compute_pynance_metrics)
        .reset_index(level=1, drop=True)
        .reset_index()
    )
else:
    metrics = compute_pynance_metrics(indicators)

metrics

## 4. Visualize Indicator Relationships

The plots below focus on one ticker at a time. When the dataset contains multiple tickers, change `selected_stock` to inspect another symbol.

In [ ]:
if 'stock' in indicators.columns:
    selected_stock = indicators['stock'].value_counts().index[0]
    plot_data = indicators[indicators['stock'] == selected_stock].copy()
else:
    selected_stock = 'Selected Stock'
    plot_data = indicators.copy()

plot_data = plot_data.sort_values('date')
selected_stock, plot_data.shape

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(plot_data['date'], plot_data['adj_close'], label='Adj Close', color='#2F4B7C', linewidth=1.8)
for column, color in [('sma_20', '#F58518'), ('sma_50', '#54A24B'), ('ema_20', '#B279A2')]:
    if column in plot_data.columns:
        ax.plot(plot_data['date'], plot_data[column], label=column.upper(), linewidth=1.2, color=color)
ax.set_title(f'{selected_stock}: Closing Price with Moving Averages')
ax.set_xlabel('Date')
ax.set_ylabel('Price')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task_2_price_moving_averages.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plot_data['date'], plot_data['rsi_14'], color='#E45756', linewidth=1.5)
ax.axhline(70, color='#C44E52', linestyle='--', linewidth=1, label='Overbought: 70')
ax.axhline(30, color='#4C78A8', linestyle='--', linewidth=1, label='Oversold: 30')
ax.set_title(f'{selected_stock}: Relative Strength Index')
ax.set_xlabel('Date')
ax.set_ylabel('RSI')
ax.set_ylim(0, 100)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task_2_rsi.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_data['date'], plot_data['macd'], label='MACD', color='#2F4B7C', linewidth=1.5)
ax.plot(plot_data['date'], plot_data['macd_signal'], label='Signal', color='#F58518', linewidth=1.2)
hist_colors = ['#54A24B' if value >= 0 else '#E45756' for value in plot_data['macd_hist'].fillna(0)]
ax.bar(plot_data['date'], plot_data['macd_hist'], label='Histogram', color=hist_colors, alpha=0.45)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'{selected_stock}: MACD Momentum')
ax.set_xlabel('Date')
ax.set_ylabel('MACD')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task_2_macd.png', dpi=150)
plt.show()

## Data Preparation Summary

- Parsed trading dates and sorted the data by ticker/date.
- Converted OHLCV columns to numeric types and used `adj_close` for return calculations.
- Checked missing values before and after cleaning.
- Forward/backward filled numeric gaps within each ticker, then removed rows that remained unusable.
- Computed SMA/EMA windows, RSI, MACD, daily returns, cumulative return, volatility, max drawdown, and Sharpe ratio.

Data quality issues to watch for in the final dataset: missing adjusted-close values, duplicate ticker-date rows, split-adjustment mismatches, and non-trading dates introduced during joins with news sentiment.